In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
import geopandas as gpd
import jenkspy
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import seasonal_decompose
import statsmodels
from scipy.stats import pearsonr
import scipy
import prophet
import warnings
warnings.filterwarnings("ignore")
from matplotlib.font_manager import FontProperties
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


In [ ]:
font = FontProperties(family='DejaVu Sans', weight='bold')


In [ ]:
pop_fb = pd.read_csv("../../../results/measure_bias/facebook_tts/Active_population_bias_nonspatial.csv", encoding="latin1")
pop_mp = pd.read_csv("../../../results/measure_bias/phone_00_08/Active_population_bias_nonspatial.csv", encoding="latin1")
pop_mp = pop_mp.rename(columns= {"Total_active_population": "active_mp"})
pop_mp = pop_mp[['Area_code', 'Area_name', 'Total_Population', 'active_mp']]
pop_fb = pd.read_csv("../../../results/measure_bias/facebook_tts/Active_population_bias_nonspatial.csv", encoding="latin1")

pop_fb = pop_fb[['Area_code', 'Total_active_population']]
df = pop_mp.merge(pop_fb, how='left', left_on='Area_code', right_on='Area_code')
df = df.rename(columns= {"Total_active_population": "active_fb"})
df = df.rename(columns= {"Total_Population": "population"})

df = df.sort_values(by="population").reset_index(drop=True)
df = df.dropna().reset_index(drop=True)

df['coverage_mp'] = df['active_mp']/df['population']
df['coverage_fb'] = df['active_fb']/df['population']

df = df[df['coverage_fb'] <= 1].reset_index(drop=True)
df = df[df['coverage_mp'] <= 1].reset_index(drop=True)

In [ ]:
df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

# Calculate color values

viridis = plt.cm.get_cmap('Purples')
norm = plt.Normalize(0, len(df) - 1)   

for i in range(len(df)):
    color = viridis(norm(i)**9)
    ax.scatter(df.loc[i, "active_mp"], df.loc[i, "active_fb"], s=35, alpha=0.6, facecolor="None", edgecolor='rebeccapurple', zorder=i)
    ax.scatter(df.loc[i, "active_mp"], df.loc[i, "active_fb"], s=35, alpha=0.9, c=color, edgecolor="None", zorder=i)

ax.plot(np.linspace(1,100000), np.linspace(1,100000), ls=":", dashes=(1, .8), linewidth=4, alpha=1, c='k')


ax.set_xlim([0.4, 200000])
ax.set_ylim([0.4, 800000])

ax.set_xscale("log")
ax.set_yscale("log")

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)
    
for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)

# plt.savefig('../plots/active-fb-mp.pdf', bbox_inches='tight')

plt.show()


In [ ]:
corr, p_value = pearsonr(df["active_mp"], df["active_fb"])
print(corr, p_value)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

# Calculate color values
viridis = plt.cm.get_cmap('Purples')
norm = plt.Normalize(0, len(df) - 1)   

for i in range(len(df)):
    color = viridis(norm(i)**9)
    ax.scatter(df.loc[i, "coverage_mp"], df.loc[i, "coverage_fb"], s=35, alpha=0.6, facecolor="None", edgecolor='rebeccapurple', zorder=i)
    ax.scatter(df.loc[i, "coverage_mp"], df.loc[i, "coverage_fb"], s=35, alpha=0.9, c=color, edgecolor="None", zorder=i)

def power_func(x, a, b):
    return a * x**b
xdata = df["coverage_mp"].values
ydata = df["coverage_fb"].values
popt, pcov = curve_fit(power_func, xdata, ydata, p0=[1, 1])
a_hat, b_hat = popt
fit = power_func(np.linspace(0.00008, max(xdata)), a_hat, b_hat)
ax.plot(np.linspace(0.00008, max(xdata)), fit, ls="-", alpha=1, lw=4, c='purple', zorder=i+1)

ax.plot(np.linspace(0.00008, max(xdata)), np.linspace(0.00008, max(xdata)), ls=":", dashes=(1, .8), alpha=1, lw=4, c='k')

ax.set_xscale("log")
ax.set_yscale("log")

# ax.set_xlim([-1800, max(df["active_mp"])+25000])
ax.set_xlim([0.00005, 0.05])
ax.set_ylim([0.00005, 2])

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)
    
for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)

# plt.savefig('../plots/coverage-fb-mp.pdf', bbox_inches='tight')

plt.show()

In [ ]:
print(a_hat, b_hat)

In [ ]:
corr, p_value = pearsonr(df["coverage_mp"], df["coverage_fb"])
print(corr, p_value)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

df = df.sort_values(by = "population").reset_index(drop=True)

viridis = plt.cm.get_cmap('Blues')
norm = plt.Normalize(0, len(df) - 1)   
for i in range(len(df)):
    color = viridis(norm(i)**9)
    ax.scatter(df.loc[i, "population"], df.loc[i, "coverage_mp"], s=35, alpha=0.6, facecolor="None", edgecolor='darkblue', zorder=i)
    ax.scatter(df.loc[i, "population"], df.loc[i, "coverage_mp"], s=35, alpha=0.9, c=color, edgecolor="None", zorder=i)

xdata = df["population"].values
ydata = df["coverage_mp"].values
popt, pcov = curve_fit(power_func, xdata, ydata, p0=[1, 1])
a_hat, b_hat = popt
fit = power_func(df["population"], a_hat, b_hat)
ax.plot(df["population"], fit, ls="-", alpha=1, lw=4, c='darkblue', zorder=i+1)
print(a_hat, b_hat)

#---

viridis = plt.cm.get_cmap('Reds')
norm = plt.Normalize(0, len(df) - 1)   
for i in range(len(df)):
    color = viridis(norm(i)**9)
    ax.scatter(df.loc[i, "population"], df.loc[i, "coverage_fb"], s=35, alpha=0.6, facecolor="None", edgecolor='crimson', zorder=i)
    ax.scatter(df.loc[i, "population"], df.loc[i, "coverage_fb"], s=35, alpha=0.9, c=color, edgecolor="None", zorder=i)

xdata = df["population"].values
ydata = df["coverage_fb"].values
popt, pcov = curve_fit(power_func, xdata, ydata, p0=[1, 1])
a_hat, b_hat = popt
fit = power_func(df["population"], a_hat, b_hat)
ax.plot(df["population"], fit, ls="-", alpha=1, lw=4, c='crimson', zorder=i+1)
print(a_hat, b_hat)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim([110, 3000000])
ax.set_ylim([0.00005, 2])

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)

for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)

# plt.savefig('../plots/scaling-coverage-pop.pdf', bbox_inches='tight')

plt.show()

In [ ]:
corr, p_value = pearsonr(df["population"], df["coverage_mp"])
print(corr, p_value)

corr, p_value = pearsonr(df["population"], df["coverage_fb"])
print(corr, p_value)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

viridis = plt.cm.get_cmap('Blues')
norm = plt.Normalize(0, len(df) - 1) 
df = df.sort_values(by = "coverage_mp", ascending=False).reset_index(drop=True)
df['rank_mp'] = np.arange(1,len(df)+1)
df = df.sort_values(by = "population", ascending=True).reset_index(drop=True)
for i in range(len(df)):
    color = viridis(norm(i)**9)
    ax.scatter(df.loc[i, "rank_mp"], df.loc[i, "coverage_mp"], s=50, alpha=0.5, facecolor="None", edgecolor='darkblue', zorder=i)
    ax.scatter(df.loc[i, "rank_mp"], df.loc[i, "coverage_mp"], s=50, alpha=0.9, c=color, edgecolor="None", zorder=i)
x = np.arange(len(df))
function_ref = df["coverage_mp"].max() / x**0.32  
# ax.plot(x, function_ref, '-', linewidth=3, alpha=0.8, color='darkblue')


viridis = plt.cm.get_cmap('Reds')
norm = plt.Normalize(0, len(df) - 1) 
df = df.sort_values(by = "coverage_fb", ascending=False).reset_index(drop=True)
df['rank_fb'] = np.arange(1,len(df)+1)
df = df.sort_values(by = "population", ascending=True).reset_index(drop=True)
for i in range(len(df)):
    color = viridis(norm(i)**9)
    ax.scatter(df.loc[i, "rank_fb"], df.loc[i, "coverage_fb"], s=50, alpha=0.5, facecolor="None", edgecolor='crimson', zorder=i)
    ax.scatter(df.loc[i, "rank_fb"], df.loc[i, "coverage_fb"], s=50, alpha=0.9, c=color, edgecolor="None", zorder=i)
x = np.arange(len(df))
function_ref = df["coverage_fb"].max() / x**0.45  
# ax.plot(x, function_ref, '-', linewidth=3, alpha=0.8, color='crimson')

ax.set_xscale("log")
ax.set_yscale("log")

# ax.set_xlim([-1800, max(df["active_mp"])+25000])
ax.set_xlim([0.5, 6000])
ax.set_ylim([0.00005, 2])

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)

for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)

# plt.savefig('../plots/size-rank-fb-mp.pdf', bbox_inches='tight')

plt.show()

In [ ]:
df

In [ ]:
print(max(df['coverage_mp']), max(df['coverage_fb']))

In [ ]:
def model_func(x, k, p):
    """Your functional form."""
    return (1 - k) * x**p + k * (1 - (1 - x)**(1 / p))


def fit_with_bands(
    x, y,
    n_samples=6000,
    include_noise=True
):
    """
    Fit y ~ model_func(x, k, p) and draw an approximate
    Bayesian (Laplace) band using the curve_fit covariance.
    """
    # Initial guess and bounds (tweak if you know better)
    p0     = [0.2, 1.5]         # k, p
    bounds = ([0.0, 0.1], [1.0, 5.0])  # k in [0,1], p in [0.1, 5]

    # Fit
    popt, pcov = curve_fit(model_func, x, y, p0=p0, bounds=bounds)
    k_hat, p_hat = popt

    # Residuals and noise estimate
    y_hat = model_func(x, k_hat, p_hat)
    resid = y - y_hat
    dof = len(y) - len(popt)
    sigma_hat = np.sqrt(np.sum(resid**2) / max(dof, 1))

    # Posterior over parameters ≈ N(popt, pcov)
    # (Laplace approximation)
    param_samples = np.random.multivariate_normal(
        mean=popt, cov=pcov, size=n_samples
    )

    # Enforce bounds by clipping
    k_samps = np.clip(param_samples[:, 0], 0.0, 1.0)
    p_samps = np.clip(param_samples[:, 1], 0.1, 5.0)
    
    # Transform parameter samples into Gini samples
    gini_samps = (p_samps - 1) / (p_samps + 1)

    # Summary statistics
    gini_hat = np.mean(gini_samps)
    gini_med = np.median(gini_samps)
    gini_lo  = np.percentile(gini_samps, 2.5)
    gini_hi  = np.percentile(gini_samps, 97.5)
    gini_se  = np.std(gini_samps, ddof=1)
    
    p_value = 2 * min(
        np.mean(gini_samps >= 0),
        np.mean(gini_samps <= 0)
    )

    # Grid for smooth curve
    x_plot = np.linspace(x.min(), x.max(), 200)
    y_draws = np.zeros((n_samples, len(x_plot)))

    for i in range(n_samples):
        k_i = k_samps[i]
        p_i = p_samps[i]
        mu_i = model_func(x_plot, k_i, p_i)

        if include_noise:
            y_draws[i, :] = np.random.normal(mu_i, sigma_hat)
        else:
            y_draws[i, :] = mu_i

    # Pointwise summaries (95% band)
    y_med = np.median(y_draws, axis=0)
    y_lo  = np.percentile(y_draws, 0.5, axis=0)
    y_hi  = np.percentile(y_draws, 99.5, axis=0)


    return k_hat, p_hat, x_plot, y_med, y_lo, y_hi, gini_hat, p_value

In [ ]:
df = df.sort_values(by="population").reset_index(drop=True)

share_pop = df['population']/np.sum(df['population'])
cum_pop = [np.sum(share_pop[:i]) for i in range(len(share_pop))]
share_mp = df['active_mp']/np.sum(df['active_mp'])
cum_active_mp = [np.sum(share_mp[:i]) for i in range(len(share_mp))]
share_fb = df['active_fb']/np.sum(df['active_fb'])
cum_active_fb = [np.sum(share_fb[:i]) for i in range(len(share_fb))]

cum_pop       = np.array(cum_pop, dtype=float)
cum_active_mp = np.array(cum_active_mp, dtype=float)
cum_active_fb = np.array(cum_active_fb, dtype=float)

# Sort by x just to keep curves monotone in x for plotting
order = np.argsort(cum_pop)
xdata = cum_pop[order]
y_mp  = cum_active_mp[order]
y_fb  = cum_active_fb[order]  # comment this out if you don't have FB data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))

# 45-degree reference line
ax.plot([xdata.min(), xdata.max()], [xdata.min(), xdata.max()], color="k", lw=4, linestyle=":", dashes=(1, .8), label="y = x")

k_hat, p_hat, x_plot, y_med, y_lo, y_hi, gini_hat_mp, p_value_gini_mp = fit_with_bands(xdata, y_mp)
ax.plot(x_plot, y_med, color="darkblue", lw=4)
ax.fill_between(x_plot, y_lo, y_hi, color="darkblue", alpha=0.2)
gini_mp = (p_hat-1)/(p_hat+1)


k_hat, p_hat, x_plot, y_med, y_lo, y_hi, gini_hat_fb, p_value_gini_fb = fit_with_bands(xdata, y_fb)
ax.plot(x_plot, y_med, color="crimson", lw=4)
ax.fill_between(x_plot, y_lo, y_hi, color="crimson", alpha=0.2)
gini_fb = (p_hat-1)/(p_hat+1)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)

for x in ax.get_xticks():
    ax.axvline(x, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
for y in ax.get_yticks():
    ax.axhline(y, color='lightgray', linewidth=0.6, linestyle='-', zorder=1)
    
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=20, pad=3)


ax.set_xlim([-0.08, 1.08])
ax.set_ylim([-0.08, 1.08])

# plt.savefig('../plots/lorenz-fb-mp.pdf', bbox_inches='tight')

plt.show()


In [ ]:
print(gini_hat_mp, p_value_gini_mp)
print(gini_hat_fb, p_value_gini_fb)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))


ax.hist(df["active_fb"], log=True, bins=100, alpha=0.5)#,  density=True)
ax.hist(df["active_mp"], log=True, bins=100, alpha=0.5)#,  density=True)

# ax.set_xscale("log")
# ax.set_yscale("log")